# Boolean Comparison and Logic Fusion

This notebook demonstrates a boolean fusion graph using the cuDNN Python API:

$$\text{output} = (\mathbf{x} > \text{threshold}) \;\wedge\; \mathbf{b}$$

The graph fuses two pointwise operations:
1. **CMP_GT** — element-wise greater-than comparison producing a boolean mask
2. **LOGICAL_AND** — element-wise AND of the comparison result with another boolean tensor

## Prerequisites and Setup

This notebook requires an NVIDIA Blackwell GPU (SM100+) or later.

**Environment setup:**

- **Option A — pip install:**
  ```bash
  pip install nvidia-cudnn-frontend
  ```
- **Option B — set paths manually:**
  ```bash
  export LD_LIBRARY_PATH=/path/to/cudnn/lib:${LD_LIBRARY_PATH}
  export PYTHONPATH=/path/to/cudnn_frontend/build_python:${PYTHONPATH}
  ```

In [1]:
# get_ipython().system('pip install nvidia-cudnn-cu13')
# get_ipython().system('pip install nvidia-cudnn-frontend')
# get_ipython().system('pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu130')

## Overview

We build a two-node fusion graph:

| Node | Operation | Inputs | Output | Compute Precision |
|------|-----------|--------|--------|-------------------|
| `cmp0` | `CMP_GT` | `x` (half), `threshold` (float) | `after_cmp` (bool, virtual) | float |
| `logical0` | `LOGICAL_AND` | `after_cmp` (bool), `b` (bool) | `output` (bool) | bool |

Tensors use row-major layout with shape `[d0, d1, d2]`.

In [2]:
import warnings
warnings.filterwarnings("ignore", message="Failed to initialize NumPy")

import torch
import cudnn

# Boolean fusion (CMP_GT + LOGICAL_AND) requires cuDNN backend version >= 92200
print("cuDNN backend version:", cudnn.backend_version())

cuDNN backend version: 92200


## Setup Tensors

We use a small 3D shape `[4, 8, 16]` in row-major layout for quick verification.

In [3]:
shape = (4, 8, 16)
has_cuda = torch.cuda.is_available()
device = torch.device("cuda" if has_cuda else "cpu")

torch.manual_seed(42)

x_gpu = torch.randn(*shape, dtype=torch.float16, device=device).contiguous()
threshold_gpu = torch.randn(*shape, dtype=torch.float32, device=device).contiguous()
b_gpu = torch.randint(0, 2, shape, dtype=torch.bool, device=device).contiguous()

print(f"x:         {x_gpu.shape}, dtype={x_gpu.dtype}, stride={x_gpu.stride()}")
print(f"threshold: {threshold_gpu.shape}, dtype={threshold_gpu.dtype}, stride={threshold_gpu.stride()}")
print(f"b:         {b_gpu.shape}, dtype={b_gpu.dtype}, stride={b_gpu.stride()}")

x:         torch.Size([4, 8, 16]), dtype=torch.float16, stride=(128, 16, 1)
threshold: torch.Size([4, 8, 16]), dtype=torch.float32, stride=(128, 16, 1)
b:         torch.Size([4, 8, 16]), dtype=torch.bool, stride=(128, 16, 1)


## Build and Execute cuDNN Graph

We use the low-level `pygraph` API to construct the fusion graph with `cmp_gt` and `logical_and` nodes.

In [4]:
if has_cuda:
    handle = cudnn.create_handle()

    graph = cudnn.pygraph(
        handle=handle,
        intermediate_data_type=cudnn.data_type.FLOAT,
        compute_data_type=cudnn.data_type.FLOAT,
    )

    x_cudnn = graph.tensor_like(x_gpu)
    threshold_cudnn = graph.tensor_like(threshold_gpu)
    b_cudnn = graph.tensor_like(b_gpu)

    after_cmp = graph.cmp_gt(
        name="cmp0",
        input=x_cudnn,
        comparison=threshold_cudnn,
        compute_data_type=cudnn.data_type.FLOAT,
    )
    after_cmp.set_data_type(cudnn.data_type.BOOLEAN)

    after_and = graph.logical_and(
        name="logical0",
        a=after_cmp,
        b=b_cudnn,
        compute_data_type=cudnn.data_type.BOOLEAN,
    )
    after_and.set_output(True).set_data_type(cudnn.data_type.BOOLEAN)

    graph.validate()
    graph.build_operation_graph()
    graph.create_execution_plans([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])
    graph.check_support()
    graph.build_plans()

    after_and_gpu = torch.empty(*shape, dtype=torch.bool, device=device).contiguous()
    workspace = torch.empty(graph.get_workspace_size(), device=device, dtype=torch.uint8)

    graph.execute(
        {x_cudnn: x_gpu, threshold_cudnn: threshold_gpu, b_cudnn: b_gpu, after_and: after_and_gpu},
        workspace,
        handle=handle,
    )
    torch.cuda.synchronize()

    print(f"after_and_gpu: {after_and_gpu.shape}, dtype={after_and_gpu.dtype}")
    print(f"True count:    {after_and_gpu.sum().item()} / {after_and_gpu.numel()}")
else:
    print("Skipping cuDNN graph (no CUDA device).")

after_and_gpu: torch.Size([4, 8, 16]), dtype=torch.bool


True count:    114 / 512


## Verify Correctness

In [5]:
if has_cuda:
    after_and_ref = (x_gpu.float() > threshold_gpu) & b_gpu
    mismatches = (after_and_gpu != after_and_ref).sum().item()
    total = after_and_ref.numel()

    print(f"Ref true count: {after_and_ref.sum().item()} / {total}")
    print(f"Mismatches:     {mismatches} / {total}")
    assert mismatches == 0, f"FAILED: {mismatches} mismatches out of {total} elements"
    print("PASSED: cuDNN boolean fusion matches reference.")
else:
    print("Skipping verification (no CUDA device).")

Ref true count: 114 / 512
Mismatches:     0 / 512
PASSED: cuDNN boolean fusion matches reference.


## Test with Multiple Shapes and Data Types

Sweep across different tensor shapes (2D–4D) and input data types for the comparison operand.

In [6]:
if has_cuda:
    test_configs = [
        {"dim": [8, 32],             "x_dtype": torch.float16},
        {"dim": [4, 8, 16],          "x_dtype": torch.float16},
        {"dim": [4, 8, 16],          "x_dtype": torch.bfloat16},
        {"dim": [4, 8, 16],          "x_dtype": torch.float32},
        {"dim": [1, 32],             "x_dtype": torch.float16},
        {"dim": [16, 64, 196],       "x_dtype": torch.float16},
        {"dim": [2, 64, 28, 28],     "x_dtype": torch.float16},
    ]

    all_pass = True
    for cfg in test_configs:
        dims = cfg["dim"]
        x_dt = cfg["x_dtype"]

        x_t = torch.randn(*dims, dtype=x_dt, device=device).contiguous()
        thresh_t = torch.randn(*dims, dtype=torch.float32, device=device).contiguous()
        b_t = torch.randint(0, 2, dims, dtype=torch.bool, device=device).contiguous()

        ref_t = (x_t.float() > thresh_t) & b_t

        g = cudnn.pygraph(
            handle=handle,
            intermediate_data_type=cudnn.data_type.FLOAT,
            compute_data_type=cudnn.data_type.FLOAT,
        )
        x_c = g.tensor_like(x_t)
        th_c = g.tensor_like(thresh_t)
        b_c = g.tensor_like(b_t)

        cmp_out = g.cmp_gt(input=x_c, comparison=th_c, compute_data_type=cudnn.data_type.FLOAT)
        cmp_out.set_data_type(cudnn.data_type.BOOLEAN)

        and_out = g.logical_and(a=cmp_out, b=b_c, compute_data_type=cudnn.data_type.BOOLEAN)
        and_out.set_output(True).set_data_type(cudnn.data_type.BOOLEAN)

        try:
            g.validate()
            g.build_operation_graph()
            g.create_execution_plans([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])
            g.check_support()
            g.build_plans()
        except cudnn.cudnnGraphNotSupportedError as e:
            print(f"[SKIP] dim={dims} x_dtype={x_dt}: {e}")
            continue

        out_t = torch.empty(*dims, dtype=torch.bool, device=device).contiguous()
        ws = torch.empty(g.get_workspace_size(), device=device, dtype=torch.uint8)
        g.execute({x_c: x_t, th_c: thresh_t, b_c: b_t, and_out: out_t}, ws, handle=handle)
        torch.cuda.synchronize()

        mm = (out_t != ref_t).sum().item()
        status = "PASS" if mm == 0 else "FAIL"
        if mm != 0:
            all_pass = False
        print(f"[{status}] dim={str(dims):20s} x_dtype={str(x_dt):18s} mismatches={mm}")
        assert mm == 0, f"Test failed for {cfg}"

    print(f"\nAll tests passed!" if all_pass else "\nSome tests failed.")
else:
    print("Skipping multi-config tests (no CUDA device).")

[PASS] dim=[8, 32]              x_dtype=torch.float16      mismatches=0
[PASS] dim=[4, 8, 16]           x_dtype=torch.float16      mismatches=0


[PASS] dim=[4, 8, 16]           x_dtype=torch.bfloat16     mismatches=0
[PASS] dim=[4, 8, 16]           x_dtype=torch.float32      mismatches=0
[PASS] dim=[1, 32]              x_dtype=torch.float16      mismatches=0


[PASS] dim=[16, 64, 196]        x_dtype=torch.float16      mismatches=0
[PASS] dim=[2, 64, 28, 28]      x_dtype=torch.float16      mismatches=0

All tests passed!


In [7]:
if has_cuda:
    cudnn.destroy_handle(handle)